# Detect DeepFake Images
## Train a MobileNetV3 binary classifier in PyTorch

In [ ]:
import pandas as pd
import numpy as np

## 1. Verify Dataset Structure

Inspect the folders under `data/Train`, `data/Validation`, and `data/Test` to ensure each has `Fake/` and `Real/` subdirectories and count the images.

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data")
TRAIN_DIR = DATA_DIR / "Train"
VAL_DIR = DATA_DIR / "Validation"
TEST_DIR = DATA_DIR / "Test"

for split in ["Train", "Validation", "Test"]:
    split_dir = DATA_DIR / split
    print(f"{split}:")
    for label in ["Fake", "Real"]:
        label_dir = split_dir / label
        count = len(list(label_dir.glob("*"))) if label_dir.exists() else 0
        print(f"  {label}: {count} images")
print("Dataset structure verified")

## 2. Setup

Import libraries, configure the device (M3 Pro/MPS or CPU), set seeds, and ensure output directories exist.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms, models
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
)
import json
from pathlib import Path
from PIL import Image
import warnings

warnings.filterwarnings("ignore")

np.random.seed(42)
torch.manual_seed(42)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

MODEL_DIR = Path("../models")
RESULTS_DIR = Path("../results")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model directory: {MODEL_DIR.resolve()}")
print(f"Results directory: {RESULTS_DIR.resolve()}")

## 3. Data Pipeline

Define `DeepfakeDataset` class and augmentations/transforms.

In [ ]:
class DeepfakeDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.images = []
        self.labels = []
        for label, idx in {"Fake": 0, "Real": 1}.items():
            dirp = self.root_dir / label
            if dirp.exists():
                for p in sorted(dirp.glob("*")):
                    if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]:
                        self.images.append(str(p))
                        self.labels.append(idx)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


IMG_SIZE = 256
BATCH_SIZE = 32
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation(15),
        transforms.ColorJitter(0.2, 0.2, 0.2),
        transforms.RandomHorizontalFlip(0.5),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ]
)

val_test_transforms = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ]
)

print("Transforms and dataset class defined")

## 4. DataLoaders

Instantiate datasets and DataLoaders.

In [ ]:
train_dataset = DeepfakeDataset(TRAIN_DIR, train_transforms)
val_dataset = DeepfakeDataset(VAL_DIR, val_test_transforms)
test_dataset = DeepfakeDataset(TEST_DIR, val_test_transforms)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

print(f"Train images: {len(train_dataset)}, batches: {len(train_loader)}")
print(f"Val images:   {len(val_dataset)}, batches: {len(val_loader)}")
print(f"Test images:  {len(test_dataset)}, batches: {len(test_loader)}")

## 5. Model Definition

Load pretrained MobileNetV3 and adapt for binary classification.

In [ ]:
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
num_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(num_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
# removed verbose argument as newer PyTorch versions drop it
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")
print("Model ready")

## 6. Training Loop

Run epochs with mixed precision, early stopping, and checkpointing.

In [ ]:
NUM_EPOCHS = 30
EARLY_STOP = 5
CHECKPOINT = MODEL_DIR / "best_model.pt"

history = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": [],
    "train_f1": [],
    "val_f1": [],
}

best_val = float("inf")
patience = 0


def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    return acc, prec, rec, f1


def run_epoch(loader, training=True):
    if training:
        model.train()
    else:
        model.eval()
    total = 0
    allp, alll = [], []
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        if training:
            optimizer.zero_grad()
        with torch.autocast(
            device_type="mps" if device.type == "mps" else device.type,
            enabled=device.type != "cpu",
        ):
            outs = model(imgs)
            loss = criterion(outs, lbls)
        if training:
            loss.backward()
            optimizer.step()
        total += loss.item()
        preds = outs.argmax(1)
        allp.extend(preds.cpu().numpy())
        alll.extend(lbls.cpu().numpy())
    return total / len(loader), *compute_metrics(alll, allp)


print("Beginning training")
for epoch in range(NUM_EPOCHS):
    t_loss, t_acc, _, _, t_f1 = run_epoch(train_loader, True)
    v_loss, v_acc, _, _, v_f1 = run_epoch(val_loader, False)
    history["train_loss"].append(t_loss)
    history["val_loss"].append(v_loss)
    history["train_acc"].append(t_acc)
    history["val_acc"].append(v_acc)
    history["train_f1"].append(t_f1)
    history["val_f1"].append(v_f1)
    print(
        f"Epoch {epoch+1}: t_loss={t_loss:.4f}, v_loss={v_loss:.4f}, v_acc={v_acc:.4f}"
    )
    scheduler.step(v_loss)
    if v_loss < best_val:
        best_val = v_loss
        patience = 0
        torch.save(model.state_dict(), CHECKPOINT)
        print("  saved best model")
    else:
        patience += 1
        if patience >= EARLY_STOP:
            print("early stopping")
            break
print("Training finished")

## 7. Evaluation on Test Set

Load saved best model and compute metrics on test data.

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT, map_location=device))
model.eval()
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs = imgs.to(device)
        out = model(imgs)
        probs = torch.softmax(out, 1)
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(lbls.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

test_acc = accuracy_score(all_labels, all_preds)
test_prec = precision_score(all_labels, all_preds, zero_division=0)
test_rec = recall_score(all_labels, all_preds, zero_division=0)
test_f1 = f1_score(all_labels, all_preds, zero_division=0)
test_auc = roc_auc_score(all_labels, all_probs[:, 1])
cm = confusion_matrix(all_labels, all_preds)

print(
    f"Accuracy {test_acc:.4f}  Precision {test_prec:.4f}  Recall {test_rec:.4f}  F1 {test_f1:.4f}  AUC {test_auc:.4f}"
)
print("Confusion matrix:\n", cm)

## 8. Visualizations

Plot training curves, confusion matrix, ROC curve, and metrics.

In [ ]:
sns.set_style("whitegrid")
fig = plt.figure(figsize=(16, 10))

plt.subplot(2, 3, 1)
plt.plot(history["train_loss"], label="Train", linewidth=2)
plt.plot(history["val_loss"], label="Val", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 3, 2)
plt.plot(history["train_acc"], label="Train", linewidth=2)
plt.plot(history["val_acc"], label="Val", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Acc")
plt.title("Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 3, 3)
plt.plot(history["train_f1"], label="Train", linewidth=2)
plt.plot(history["val_f1"], label="Val", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("F1")
plt.title("F1 Score")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 3, 4)
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Fake", "Real"],
    yticklabels=["Fake", "Real"],
)
plt.title("Confusion Matrix")

plt.subplot(2, 3, 5)
fpr, tpr, _ = roc_curve(all_labels, all_probs[:, 1])
plt.plot(fpr, tpr, label=f"ROC AUC={test_auc:.4f}", linewidth=2)
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 3, 6)
met = [test_acc, test_prec, test_rec, test_f1, test_auc]
names = ["Acc", "Prec", "Rec", "F1", "AUC"]
bars = plt.bar(names, met)
plt.ylim(0, 1)
for b, h in zip(bars, met):
    plt.text(b.get_x() + b.get_width() / 2, h, f"{h:.3f}", ha="center", va="bottom")
plt.title("Test Metrics")
plt.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Save Results

Write metrics and training history to JSON files.

In [ ]:
metrics = {
    "model": "MobileNetV3-Small",
    "device": str(device),
    "test": {
        "accuracy": float(test_acc),
        "precision": float(test_prec),
        "recall": float(test_rec),
        "f1": float(test_f1),
        "auc": float(test_auc),
    },
    "confusion_matrix": {
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
    },
    "sizes": {
        "train": len(train_dataset),
        "val": len(val_dataset),
        "test": len(test_dataset),
    },
}
with open(RESULTS_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
with open(RESULTS_DIR / "history.json", "w") as f:
    json.dump(history, f, indent=2)
print(f"Saved metrics to {RESULTS_DIR}")

## 10. Summary

Training and evaluation complete — check `models/best_model.pt` and `results/` for outputs.